# Behavioral: AWS Serverless Migration Story

| Story | Core Signal |
|-------|-------------|
| Serverless Migration | Cloud-native architecture, cost reduction |
| Lambda + DynamoDB + Kinesis | Specific service choices with trade-offs |
| Rollout Strategy | Phased migration, zero-downtime cutover |
| Failure Modes | Cold start, DLQ handling, throughput limits |
| Follow-up Defense | Deep technical questions on serverless design |

**Purpose**: A serverless migration story demonstrates cloud architecture decision-making, understanding of managed-service trade-offs, and production operations experience.

```
Core technical signals:
  Lambda: event-driven, stateless, cold start trade-off
  DynamoDB: single-table design, access patterns drive schema
  Kinesis: ordered, partitioned stream; consumer fanout
  SQS: queue with DLQ; at-least-once delivery
  Migration: strangler fig pattern — route traffic incrementally
```

## Visual Model

```
BEFORE (monolith)
──────────────────
  EC2 instance (always-on, over-provisioned)
  ├── REST API (Flask)
  ├── Background workers (Celery + Redis)
  ├── Cron jobs (crontab)
  └── PostgreSQL (RDS, single instance)
  Cost: ~$8,400/month. Scaling = manual capacity planning.

AFTER (serverless)
───────────────────
  API Gateway → Lambda (per endpoint, stateless)
  Kinesis Data Streams → Lambda consumers (event pipeline)
  EventBridge → Lambda (cron replacement)
  DynamoDB (single-table, on-demand capacity)
  S3 + Athena (analytics queries, cold storage)
  Cost: ~$1,100/month (87% reduction, traffic-proportional)

MIGRATION STRATEGY (strangler fig)
────────────────────────────────────
  Week 1-2:  shadow mode — new Lambda duplicates requests, logs only
  Week 3:    route 10% of low-risk endpoints to Lambda
  Week 4:    50% of traffic
  Week 5:    100% of traffic, old EC2 on standby
  Week 6:    decommission EC2 after 1-week clean observation
```

## Setup — Story Framework

In [ ]:
def print_story(title, situation, task, actions, result, timing_s=120):
    print(f"\n{'='*60}")
    print(f"STORY: {title}")
    print(f"Target: ~{timing_s}s verbal delivery")
    print(f"{'='*60}")
    print("\n[SITUATION] ~20s")
    print(situation)
    print("\n[TASK] ~10s")
    print(task)
    print("\n[ACTIONS] ~60s")
    for i, action in enumerate(actions, 1):
        print(f"  {i}. {action}")
    print("\n[RESULT] ~30s")
    print(result)

def print_followup(q, a):
    print(f"\nQ: {q}")
    print(f"A: {a}")

print("Story framework loaded.")

## Decision Map — Serverless Architecture

```
Architecture decision → service selection
│
├─ HTTP API with variable traffic?  → API Gateway + Lambda
│   Trade-off: cold start (~100-500ms on first invoke)
│   Mitigation: provisioned concurrency for latency-sensitive paths
│
├─ Event-driven processing?         → Kinesis (ordered) or SQS (unordered)
│   Kinesis: per-shard ordering, 7-day retention, fan-out consumers
│   SQS:     simpler, at-least-once, easier DLQ, no strict ordering
│
├─ Key-value / low-latency reads?   → DynamoDB
│   Design: single-table, access patterns first, GSIs for alternate keys
│   Avoid: large scans, ad-hoc queries (use Athena + S3 for those)
│
├─ Scheduled batch jobs?            → EventBridge + Lambda
│   Replaces: crontab, scheduled EC2 tasks
│
├─ Analytics / SQL?                 → S3 + Athena (or Redshift Serverless)
│   Export DynamoDB streams → S3 → Athena for ad-hoc analysis
│
└─ Migration approach?
    Strangler fig: route traffic incrementally
    Shadow mode:   run new path in parallel, compare outputs
    Feature flags: toggle per endpoint/user
    Never: big bang migration (high risk, hard rollback)
```

## Pattern 1 — Serverless Migration: Core Story

In [ ]:
print_story(
    title="AWS Serverless Migration — Monolith to Event-Driven",

    situation="""
At a fintech startup where I was the senior data engineer, we had a
monolithic Flask application running on over-provisioned EC2 instances
costing ~$8,400/month. Traffic was highly bursty — near-zero at night,
heavy bursts during business hours — but we were paying for peak capacity
24/7. The ops team spent 4-6 hours per week on manual scaling events
and one outage had cost us a key enterprise customer.
""",

    task="""
I proposed and owned the migration from the monolith to a serverless
architecture. I was responsible for architecture design, sequencing the
migration to minimize risk, and training the rest of the team on
Lambda and DynamoDB patterns.
""",

    actions=[
        """Started with an access pattern audit of the PostgreSQL database.
   The monolith had 230 tables but 80% of API traffic hit 7 access patterns.
   I modeled those 7 patterns in DynamoDB single-table design,
   which kept lookup latency under 5ms P99 vs 80ms avg on RDS.""",

        """Replaced the Celery+Redis event processing with Kinesis Data Streams.
   I chose Kinesis over SQS because we needed ordered processing
   per customer_id — financial transactions must be applied in sequence.
   Each shard handles one partition of customer IDs; Lambda consumers
   checkpoint to DynamoDB after each batch.""",

        """Used a strangler fig migration: first ran Lambda in shadow mode —
   receiving the same requests, logging outputs but not serving responses.
   After 2 weeks of output diff validation, I routed 10% → 50% → 100%
   of traffic over 3 weeks with feature flags per endpoint type.
   This gave us clean rollback at every step.""",

        """Addressed Lambda cold starts by enabling provisioned concurrency
   on the 3 latency-sensitive endpoints (account balance, transaction submit,
   auth). Cold start penalty on those was 400ms — unacceptable for our
   <200ms SLA. The other 20 endpoints used on-demand concurrency since
   they were background operations.""",
    ],

    result="""
Monthly compute cost: $8,400 → $1,100 (87% reduction).
Scaling: fully automatic — handled 40x traffic spike on launch day
  without any ops intervention.
P99 API latency: 380ms → 95ms on the migrated endpoints.
Zero downtime during migration. The ops team recovered 4-6 hours/week
  previously spent on capacity management.
The enterprise customer that had churned came back 3 months later
  after hearing about the architecture changes.
""",
    timing_s=150
)

## Pattern 2 — DynamoDB Design Story

In [ ]:
# Demonstrate DynamoDB single-table design thinking
# This is often a follow-up or a standalone system design question

print("""DynamoDB Single-Table Design — Key Concepts
════════════════════════════════════════════

Access patterns drive schema (not the reverse):
  Access pattern 1: Get customer by customer_id
    PK = CUSTOMER#<id>      SK = METADATA
  Access pattern 2: Get all orders for a customer
    PK = CUSTOMER#<id>      SK = ORDER#<order_date>#<order_id>
  Access pattern 3: Get order by order_id
    GSI: PK = ORDER#<id>    SK = CUSTOMER#<id>
  Access pattern 4: Get recent orders (last 30 days)
    PK = CUSTOMER#<id>      SK begins_with ORDER#2024-03

Key selection rules:
  PK (partition key): determines shard — choose high-cardinality for even distribution
  SK (sort key):      enables range queries within a partition
  Avoid: customer_id as PK if one customer has millions of orders (hot partition)
  Fix:   shard the PK: CUSTOMER#<id>#<shard_0..9>

Capacity modes:
  On-demand:    pay per request, auto-scales — best for unpredictable traffic
  Provisioned:  set read/write capacity units, cheaper at steady load
  Auto-scaling: provisioned but with CloudWatch-triggered adjustment

DLQ for Lambda consumer:
  Lambda reads Kinesis in batches
  On failure: retry (configurable), then poison pill → dead-letter queue
  DLQ: SQS queue with alarm on depth > 0
  Process DLQ manually or with separate Lambda for inspection/retry
""")

## Pattern 3 — Follow-up Defense: Serverless

In [ ]:
followups = [
    (
        "Why Kinesis over SQS for the event pipeline?",
        """Two reasons. First, ordering: financial transactions for a given customer
must be applied in the order they arrive. SQS standard queues don't guarantee
order. SQS FIFO does, but it's limited to 3,000 msgs/sec, which we'd hit
during peak. Second, fan-out: we had three consumers — the core processor,
a fraud detection service, and an audit logger. Kinesis lets multiple
consumer groups read the same stream independently. With SQS you'd need
three separate queues and a fan-out SNS in front."""
    ),
    (
        "What happens if a Lambda consumer falls behind on Kinesis?",
        """Kinesis retains data for 7 days by default (up to 365 with extended
retention). If a consumer Lambda is throttled or hitting errors, the iterator
position falls behind. This creates head-of-line blocking within a shard —
the shard can't advance past the failed batch.
Our mitigation: set bisect_batch_on_function_error=True so a failing batch
is split in half to isolate poison pills. Failed records go to SQS DLQ.
We had a CloudWatch alarm on DLQ depth > 0 that paged on-call within 1 minute."""
    ),
    (
        "How did you handle DynamoDB hot partitions?",
        """We identified two hot partition risks. One was our most active customer
accounts — we added a shard suffix (0-9) to their partition key and
round-robined writes. Reads scatter-gathered across all 10 shards.
The second was a time-based access pattern (today's transactions) —
we added DAX (DynamoDB Accelerator) in front, which cached the hot
read path and absorbed 90% of read load from DynamoDB itself."""
    ),
    (
        "What did you do for observability after the migration?",
        """Three layers:
1. Lambda: structured JSON logs → CloudWatch Logs → Log Insights queries
   for error rate, latency percentiles per function
2. Custom metrics: emitted via EMF (Embedded Metrics Format) — zero overhead,
   avoids PutMetricData API costs
3. X-Ray distributed tracing: end-to-end request trace from API Gateway →
   Lambda → DynamoDB, which let us see exactly where latency lived.
We also had a synthetic canary (CloudWatch Synthetics) hitting each
critical endpoint every 60 seconds from 3 regions."""
    ),
    (
        "Would you do anything differently?",
        """I'd implement the DynamoDB data model before migrating the application code,
not concurrently. We had a period where the data model was still being refined
while engineers were writing Lambda handlers against it — that caused two
breaking schema changes mid-migration. The rule should be: finalize access
patterns → lock the data model → then write application code.
Also, I'd instrument cost anomaly detection from day one.
Serverless costs are hard to predict with new traffic patterns —
we had one week where a bug caused 100x Lambda invocations and cost
us $400 extra before we noticed."""
    ),
]

for q, a in followups:
    print_followup(q, a)

## Pattern 4 — Lambda + Kinesis Architecture Simulation

In [ ]:
# Simulate key serverless patterns in Python
# Demonstrates understanding of how these services work

import random
import hashlib
from collections import defaultdict, deque

class KinesisStream:
    """Simulates a Kinesis stream: partitioned, ordered within shard."""

    def __init__(self, name, num_shards=4):
        self.name = name
        self.num_shards = num_shards
        self.shards = [deque() for _ in range(num_shards)]
        self.sequence_counters = [0] * num_shards

    def put_record(self, partition_key, data):
        """Route record to shard based on partition key hash."""
        shard_id = int(hashlib.md5(partition_key.encode()).hexdigest(), 16) % self.num_shards
        seq = self.sequence_counters[shard_id]
        self.sequence_counters[shard_id] += 1
        self.shards[shard_id].append({"partition_key": partition_key, "data": data, "sequence": seq})
        return shard_id, seq

    def get_records(self, shard_id, limit=10):
        """Consumer reads batch from a shard."""
        batch = []
        for _ in range(min(limit, len(self.shards[shard_id]))):
            batch.append(self.shards[shard_id].popleft())
        return batch


class LambdaConsumer:
    """Simulates a Lambda function consuming from Kinesis with DLQ."""

    def __init__(self, name, fail_rate=0.05):
        self.name = name
        self.fail_rate = fail_rate
        self.processed = 0
        self.dlq = []

    def process_batch(self, records, bisect=True):
        """Process a batch; on failure bisect (like bisect_batch_on_function_error)."""
        if not records:
            return
        # Simulate random failure
        if random.random() < self.fail_rate:
            if len(records) == 1:
                print(f"  [{self.name}] POISON PILL → DLQ: seq={records[0]['sequence']}")
                self.dlq.append(records[0])
                return
            if bisect:
                mid = len(records) // 2
                print(f"  [{self.name}] Batch fail → bisecting {len(records)} → {mid} + {len(records)-mid}")
                self.process_batch(records[:mid], bisect=True)
                self.process_batch(records[mid:], bisect=True)
                return
        for r in records:
            self.processed += 1


# Simulate financial transaction stream
random.seed(42)
stream = KinesisStream("transactions", num_shards=4)
consumer = LambdaConsumer("tx-processor", fail_rate=0.15)

# Put 30 records for 5 customers (ordered per customer within shard)
customers = [f"CUST#{i:03d}" for i in range(1, 6)]
for i in range(30):
    cust = random.choice(customers)
    shard_id, seq = stream.put_record(cust, {"amount": random.randint(10, 1000)})

print("Shard sizes:", [len(s) for s in stream.shards])
print("\nProcessing shard 0:")
records = stream.get_records(0, limit=15)
consumer.process_batch(records)

print(f"\nResults: processed={consumer.processed}, dlq={len(consumer.dlq)}")
print(f"DLQ sequences: {[r['sequence'] for r in consumer.dlq]}")

## Pattern 5 — Short-Form Variant Stories

In [ ]:
short_stories = {
    "Cold start mitigation": """
During the Lambda migration, we noticed the payment submission endpoint
had P99 latency of 950ms — well above our 200ms SLA.
Root cause: Lambda cold starts on the 128MB memory function.
I increased memory to 512MB (which also increases CPU allocation),
enabled provisioned concurrency at 10 instances,
and moved the DynamoDB client initialization outside the handler
(module-level, so it's reused across warm invocations).
P99 dropped from 950ms to 82ms. Monthly cost of provisioned concurrency:
+$45/month — trivially worth it.
""",

    "DynamoDB schema migration": """
Three months into production, the product team added a new requirement:
list all transactions for a given merchant (not just per customer).
Our single-table design didn't support this access pattern.
I added a GSI (Global Secondary Index) with PK=MERCHANT#<id> and SK=ORDER#<date>.
In DynamoDB, you can add a GSI without any downtime or schema migration —
it backfills asynchronously. The new access pattern was live in 15 minutes.
This is the advantage of the single-table design:
you extend access patterns with GSIs, not schema migrations.
""",

    "Cost spike incident": """
Two weeks after migration, I got a cost anomaly alert at 3am:
Lambda spend was 20x normal. A bug had put an infinite retry loop
in a Lambda function — each invocation triggered another invocation.
I set the function's reserved concurrency to 0 (effectively disabling it),
deployed the fix, re-enabled it, and the spike stopped.
Total extra cost: $380. We added an AWS Cost Anomaly Detection rule
and a maximum concurrency limit on all functions the next morning.
Lesson: serverless cost is code behavior, not just infrastructure config.
""",

    "Zero-downtime cutover": """
For the final cutover from EC2 to Lambda, I used Route 53 weighted routing:
started at 5% Lambda / 95% EC2, watched error rates and latency in real time,
and shifted 10% every 15 minutes over 2 hours.
When we hit 100% Lambda, I kept the EC2 instances running for 7 days
with zero traffic, then terminated them.
We had one incident: a Lambda timeout on a heavy computation that EC2
could handle in 45 seconds (Lambda max: 15 minutes, but this one was 20).
We moved that function to a Step Functions workflow with async execution.
""",
}

for label, story in short_stories.items():
    print(f"\n{'─'*50}")
    print(f"SHORT STORY: {label}")
    print(f"{'─'*50}")
    print(story.strip())

## Full Decision Map

```
SERVERLESS ARCHITECTURE DECISIONS
────────────────────────────────────
Compute:
  Lambda:            event-driven, stateless, <15 min
  Fargate:           container, long-running, stateful
  Step Functions:    orchestrate multi-step workflows, async

Events / Messaging:
  Kinesis:           ordered per shard, fan-out, streaming
  SQS:               at-least-once, decoupled, no ordering (std)
  SNS:               pub/sub fan-out, push to SQS/Lambda/email
  EventBridge:       event bus, routing rules, scheduled events

Data:
  DynamoDB:          key-value, <10ms, access-pattern-driven schema
  S3:                blob/object storage, event triggers
  Athena:            serverless SQL on S3, pay-per-query
  Aurora Serverless: relational, pauses when idle

Migration strategy:
  Strangler fig:     replace piece by piece, route by feature flag
  Shadow mode:       run new path in parallel, compare, no serving
  Never big bang:    all-at-once = high risk, no clean rollback

Lambda cold start mitigation:
  Provisioned concurrency: pre-warmed instances (cost: ~$0.015/hr/instance)
  Higher memory:           more CPU → faster init time
  Module-level init:       client init outside handler → reused on warm
  SnapStart (Java):        snapshot after init, restore on invoke
```

## Cheat Sheet

```
AWS SERVERLESS MIGRATION — KEY FACTS
──────────────────────────────────────
NUMBERS
  Before: $8,400/month EC2, 4-6h/week ops, 380ms P99
  After:  $1,100/month (87% reduction), auto-scale, 95ms P99
  Cold start fix: 512MB + provisioned concurrency → 950ms → 82ms

SERVICE SELECTION REASONS
  Kinesis over SQS: ordered per customer + fan-out consumers
  DynamoDB over RDS: <5ms P99, on-demand scale, no connection limits
  EventBridge over cron: native scheduling, retry, observability

MIGRATION PATTERN
  Shadow → 10% → 50% → 100% (weighted routing via Route 53)
  EC2 on standby 7 days after 100% cutover before termination

FAILURE PATTERNS + MITIGATIONS
  Cold start:      provisioned concurrency + higher memory + module-level init
  Poison pill:     bisect_batch_on_function_error + DLQ + alert
  Hot partition:   shard suffix on PK + DAX for read caching
  Cost spike:      reserved concurrency cap + cost anomaly detection
  Long-running job: Lambda 15-min limit → Step Functions async workflow

DYNAMODB PATTERN
  Single-table design: one table, composite keys, GSIs for alt access
  PK high-cardinality (avoid hot partition)
  SK enables range queries within partition
  GSI: add without downtime, async backfill
```

## Summary Map

```
AWS SERVERLESS MIGRATION — ONE-PAGE SUMMARY
─────────────────────────────────────────────

CORE STORY
  S: Flask monolith, $8.4k/month, bursty traffic, manual ops
  T: Owned architecture + migration strategy
  A: DynamoDB single-table (7 access patterns), Kinesis (ordered),
     strangler fig migration, provisioned concurrency
  R: 87% cost reduction, auto-scale, zero downtime, P99 380→95ms

KEY DECISIONS + RATIONALE
  Kinesis vs SQS:     ordering per customer + multi-consumer fan-out
  DynamoDB vs RDS:    access-pattern-first schema, <5ms, no connections
  Prov. concurrency:  $45/mo for latency-sensitive endpoints → P99 950→82ms
  Shadow mode:        2 weeks validation before routing live traffic

FAILURE STORIES
  Bisect on error:    isolate poison pills in Kinesis consumer
  Cost spike:         reserve concurrency cap + anomaly detection
  20-min Lambda:      Step Functions async for jobs > 15 min

INTERVIEW SIGNALS
  ✓ Access-pattern-driven DynamoDB design
  ✓ Kinesis vs SQS trade-off knowledge
  ✓ Cold start mitigation options
  ✓ Strangler fig migration (never big bang)
  ✓ Observability: CloudWatch + EMF + X-Ray + Synthetics
```